# Colab launcher

Set **Runtime > Change runtime type > GPU** before running. Then run the single cell below.


In [ ]:
#@title Build and run the CUDA seed tool
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/YOUR_REPO_NAME.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
LARGE_BIOMES = False #@param {type:"boolean"}
UNBOUND = True #@param {type:"boolean"}
PRINT_INTERVAL = 256 #@param {type:"integer"}
CUDA_ARCH = "auto" #@param ["auto", "sm_75", "sm_80", "sm_86", "sm_89", "sm_90"]
RUN_ARGS = "--device 0" #@param {type:"string"}
DISCORD_WEBHOOK_URL = "" #@param {type:"string"}
DISCORD_MESSAGE_PREFIX = "Seed output" #@param {type:"string"}

import os, shlex, subprocess

env = os.environ.copy()
env["LARGE_BIOMES"] = "1" if LARGE_BIOMES else "0"
env["UNBOUND"] = "1" if UNBOUND else "0"
env["PRINT_INTERVAL"] = str(PRINT_INTERVAL)
if CUDA_ARCH != "auto":
    env["CUDA_ARCH"] = CUDA_ARCH

# Prefer a pasted value in the form field. If left blank, try a Colab Secret
# named DISCORD_WEBHOOK_URL. Do not commit webhook URLs into public GitHub repos.
webhook = DISCORD_WEBHOOK_URL.strip()
if not webhook:
    try:
        from google.colab import userdata
        webhook = (userdata.get("DISCORD_WEBHOOK_URL") or "").strip()
    except Exception:
        webhook = ""

if webhook:
    env["DISCORD_WEBHOOK_URL"] = webhook
    env["DISCORD_MESSAGE_PREFIX"] = DISCORD_MESSAGE_PREFIX

cmd = f'''
set -euo pipefail
rm -rf /content/comission_cuda
git clone --depth 1 --branch {shlex.quote(BRANCH)} {shlex.quote(REPO_URL)} /content/comission_cuda
cd /content/comission_cuda/cuda_com
chmod +x ./colab_run.sh ./discord_output_bridge.py
./colab_run.sh {RUN_ARGS}
'''
subprocess.run(["bash", "-lc", cmd], env=env, check=True)
